# Project 1 — Map Reconstruction & MCQ Answering

**Kaggle setup checklist (do this before saving/submitting):**
1. Add competition dataset (provides `patches/`, `test.csv`, `sample_submission.csv`)
2. Add model: **Qwen/Qwen2-VL-7B-Instruct** from Kaggle Models Hub
   - Notebook → Add-ons → Models → search `Qwen2-VL-7B` → attach
   - Mounts at `/kaggle/input/qwen2-vl/transformers/7b-instruct/1`
3. Accelerator: **GPU (T4 x2 or P100)**
4. Internet: **OFF**

**Pipeline:**
- SIFT feature extraction → pairwise FLANN matching → RANSAC homography → BFS from `patch_0` → warp & blend canvas
- Qwen2-VL-7B (offline) answers the MCQs from the stitched map image
- Outputs `submission.csv` to `/kaggle/working/`

In [ ]:
# Cell 1 — Install missing package offline
# qwen-vl-utils is needed to process image inputs for Qwen2-VL.
# It is small and Kaggle allows pip install from cached packages.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'qwen-vl-utils', '--quiet'], check=True)
print('Dependencies ready')

In [ ]:
# Cell 2 — Imports & Paths
import os, re, math, warnings
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

warnings.filterwarnings('ignore')

# ── Competition data ────────────────────────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input')
# Auto-detect competition folder (first folder that has test.csv)
COMP_DIR  = next(
    (d for d in sorted(INPUT_DIR.iterdir()) if (d/'test.csv').exists()),
    INPUT_DIR
)
PATCHES_DIR = COMP_DIR / 'patches'
TEST_CSV    = COMP_DIR / 'test.csv'
SAMPLE_CSV  = COMP_DIR / 'sample_submission.csv'

# ── Model (Kaggle Models Hub) ───────────────────────────────────────────────────
# Try standard Kaggle mount path first, then scan for any qwen config
MODEL_DIR = Path('/kaggle/input/qwen2-vl/transformers/7b-instruct/1')
if not MODEL_DIR.exists():
    for cfg in INPUT_DIR.rglob('config.json'):
        if 'qwen' in str(cfg).lower():
            MODEL_DIR = cfg.parent
            break

# ── Outputs ─────────────────────────────────────────────────────────────────────
WORKING   = Path('/kaggle/working')
OUT_MAP   = WORKING / 'stitched_map.png'
SUBMIT    = WORKING / 'submission.csv'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Competition dir :', COMP_DIR)
print('Patches dir     :', PATCHES_DIR)
print('Model dir       :', MODEL_DIR)
print('Device          :', DEVICE)

In [ ]:
# Cell 3 — Load patches (patch_0 = top-left anchor, rest may be shuffled/rotated)
patch_files = sorted(
    PATCHES_DIR.glob('patch_*.png'),
    key=lambda p: int(re.search(r'\d+', p.stem).group())
)
patches = []
for pf in patch_files:
    img = cv2.imread(str(pf))
    if img is not None:
        patches.append((pf.stem, img))

print(f'Loaded {len(patches)} patches')
if patches:
    h, w = patches[0][1].shape[:2]
    print(f'Patch size example: {w}x{h}')

In [ ]:
# Cell 4 — SIFT feature extraction
def extract_sift(img, n_feat=2000):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=n_feat)
    return sift.detectAndCompute(gray, None)

print('Extracting SIFT features ...')
features = []
for pid, img in patches:
    kp, des = extract_sift(img)
    features.append((pid, img, kp, des))
    print(f'  {pid}: {len(kp)} keypoints')

In [ ]:
# Cell 5 — Pairwise FLANN matching + RANSAC homography
FLANN = cv2.FlannBasedMatcher({'algorithm': 1, 'trees': 5}, {'checks': 50})

def match_pair(fa, fb, ratio=0.75, min_inl=10):
    """Returns (H, n_inliers) where H maps fb-local → fa-local, or None."""
    _, _, kpA, dA = fa
    _, _, kpB, dB = fb
    if dA is None or dB is None or len(dA) < 4 or len(dB) < 4:
        return None
    try:
        raw = FLANN.knnMatch(dA, dB, k=2)
    except cv2.error:
        return None
    good = [m for m, n in raw if m.distance < ratio * n.distance]
    if len(good) < min_inl:
        return None
    src = np.float32([kpA[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([kpB[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H, mask = cv2.findHomography(dst, src, cv2.RANSAC, 5.0)
    if H is None:
        return None
    inl = int(mask.sum()) if mask is not None else 0
    return (H, inl) if inl >= min_inl else None

print('Pairwise matching ...')
n = len(features)
edges = {}   # (i,j) -> (H, inliers)
for i in range(n):
    for j in range(n):
        if i != j:
            r = match_pair(features[i], features[j])
            if r:
                edges[(i,j)] = r
print(f'{len(edges)} directed edges found')

In [ ]:
# Cell 6 — BFS from patch_0 to compute global transforms
def bfs_global_transforms(features, edges):
    n   = len(features)
    T   = {0: np.eye(3, dtype=np.float64)}   # patch_0 is the anchor
    vis = {0}
    q   = deque([0])
    adj = defaultdict(list)
    for (i,j),(H,inl) in edges.items():
        adj[i].append((j, H, inl))
    while q:
        cur = q.popleft()
        for nxt, H, inl in sorted(adj[cur], key=lambda x: -x[2]):
            if nxt not in vis:
                T[nxt] = T[cur] @ H
                vis.add(nxt)
                q.append(nxt)
    for i in range(n):
        if i not in T:
            print(f'  WARNING: {features[i][0]} unreachable — placed at origin')
            T[i] = np.eye(3, dtype=np.float64)
    print(f'BFS placed {len(vis)}/{n} patches')
    return T

transforms = bfs_global_transforms(features, edges)

In [ ]:
# Cell 7 — Compute canvas size, warp & blend
corners_all = []
for i,(pid,img,kp,des) in enumerate(features):
    h,w = img.shape[:2]
    c = np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1,1,2)
    corners_all.append(cv2.perspectiveTransform(c, transforms[i]))

pts  = np.concatenate(corners_all)
x0,y0 = pts[:,:,0].min(), pts[:,:,1].min()
x1,y1 = pts[:,:,0].max(), pts[:,:,1].max()

# Shift to positive coords
Tt = np.array([[1,0,-x0],[0,1,-y0],[0,0,1]], dtype=np.float64)
for i in transforms:
    transforms[i] = Tt @ transforms[i]

cw, ch = int(math.ceil(x1-x0)), int(math.ceil(y1-y0))

# Safety cap
MAX_DIM = 14000
if max(cw,ch) > MAX_DIM:
    s = MAX_DIM / max(cw,ch)
    S = np.diag([s,s,1.0])
    for i in transforms: transforms[i] = S @ transforms[i]
    cw, ch = int(cw*s), int(ch*s)

print(f'Canvas: {cw}x{ch}')

acc = np.zeros((ch,cw,3), dtype=np.float32)
wgt = np.zeros((ch,cw),   dtype=np.float32)
for i,(pid,img,kp,des) in enumerate(features):
    w = cv2.warpPerspective(img, transforms[i], (cw,ch)).astype(np.float32)
    m = (w.sum(axis=2) > 0).astype(np.float32)
    for c in range(3): acc[:,:,c] += w[:,:,c]*m
    wgt += m

stitched = np.clip(acc / np.maximum(wgt,1)[:,:,None], 0, 255).astype(np.uint8)
cv2.imwrite(str(OUT_MAP), stitched)
print(f'Stitched map saved → {OUT_MAP}')

In [ ]:
# Cell 8 — Fallback: OpenCV Stitcher or grid tiling if result is empty
fill = (cv2.cvtColor(stitched, cv2.COLOR_BGR2GRAY) > 0).mean()
print(f'Fill ratio: {fill:.1%}')

if fill < 0.05:
    imgs = [img for _,img,_,_ in features]
    st   = cv2.Stitcher_create(cv2.Stitcher_SCANS)
    ok, out = st.stitch(imgs)
    if ok == cv2.Stitcher_OK:
        stitched = out
        print('OpenCV Stitcher succeeded')
    else:
        nc = math.ceil(math.sqrt(len(features)))
        nr = math.ceil(len(features)/nc)
        ph,pw = features[0][1].shape[:2]
        g = np.zeros((nr*ph,nc*pw,3), dtype=np.uint8)
        for idx,(_,img,_,_) in enumerate(features):
            r,c = divmod(idx,nc)
            g[r*ph:(r+1)*ph, c*pw:(c+1)*pw] = img[:ph,:pw]
        stitched = g
        print('Grid tiling fallback used')
    cv2.imwrite(str(OUT_MAP), stitched)

In [ ]:
# Cell 9 — Load Qwen2-VL-7B from local Kaggle model mount (no internet)
print(f'Loading model from {MODEL_DIR} ...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,
)
processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True
)
model.eval()
print('Model ready')

In [ ]:
# Cell 10 — VLM inference function
def ask_map_vlm(image_path: str, question: str, opts: list) -> int:
    opt_str = '\n'.join(f'{i+1}. {o}' for i,o in enumerate(opts))
    prompt  = (
        f"{question}\n\nOptions:\n{opt_str}\n\n"
        "Look carefully at the map. Reply ONLY with the number "
        "of the correct option (1, 2, 3, or 4). "
        "If you cannot determine the answer, reply 5."
    )
    messages = [{'role':'user','content':[
        {'type':'image','image':f'file://{image_path}'},
        {'type':'text','text':prompt}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids,
                       padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    raw = processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'[1-5]', raw)
    return int(m.group()) if m else 5

In [ ]:
# Cell 11 — Answer all questions & write submission.csv
df   = pd.read_csv(TEST_CSV)
rows = []
map_path = str(OUT_MAP.resolve())

for _, row in df.iterrows():
    qid  = row['id']
    opts = [row[f'option_{i}'] for i in range(1,5)]
    pred = ask_map_vlm(map_path, row['question'], opts)
    rows.append({'id': qid, 'question_num': qid, 'option': pred})
    print(f'{qid} → {pred}')

sub = pd.DataFrame(rows, columns=['id','question_num','option'])
sub.to_csv(SUBMIT, index=False)
print(f'\nsubmission.csv written ({len(sub)} rows)')
print(sub)

In [ ]:
# Cell 12 — Validate against sample_submission columns & row count
result = pd.read_csv(SUBMIT)
sample = pd.read_csv(SAMPLE_CSV)
assert list(result.columns) == list(sample.columns), \
    f'Column mismatch: {result.columns.tolist()} vs {sample.columns.tolist()}'
assert len(result) == len(df), f'Row mismatch: {len(result)} vs {len(df)}'
assert result['option'].between(1,5).all(), 'Option values must be 1-5'
print('submission.csv is valid ✓')
print(result)